# Computer Vision aur OpenCV

Yeh notebook [AI for Beginners Curriculum](http://aka.ms/ai-beginners) ka hissa hai.

[OpenCV](https://opencv.org/) ko image processing ke liye *de facto* standard maana jaata hai. Isme kai useful algorithms hain, jo C++ mein implement kiye gaye hain. Aap OpenCV ko Python se bhi call kar sakte hain.

Is notebook mein, hum aapko OpenCV use karne ke kuch examples denge. Zyada jaankari ke liye, aap [Learn OpenCV](https://learnopencv.com/getting-started-with-opencv/) online course dekh sakte hain.

Sabse pehle, chaliye `cv2` import karte hain, saath hi kuch aur useful libraries bhi:

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

def display_images(l,titles=None,fontsize=12):
    n=len(l)
    fig,ax = plt.subplots(1,n)
    for i,im in enumerate(l):
        ax[i].imshow(im)
        ax[i].axis('off')
        if titles is not None:
            ax[i].set_title(titles[i],fontsize=fontsize)
    fig.set_size_inches(fig.get_size_inches()*n)
    plt.tight_layout()
    plt.show()


## Image Load karna

Python mein images ko aasaani se NumPy arrays ke roop mein represent kiya ja sakta hai. Jaise, 320x200 pixels ki grayscale image 200x320 array mein store hogi, aur usi dimension ki color image ka shape 200x320x3 hoga (3 color channels ke liye).

Chaliye ek image load karke shuru karte hain:

In [ ]:
im = cv2.imread('data/braille.jpeg')
print(im.shape)
plt.imshow(im)

Jaisa ki aap dekh sakte hain, yeh Braille text ki image hai. Kyunki humein actual color mein zyada interest nahi hai, hum ise black and white mein convert kar sakte hain:

In [ ]:
bw_im = cv2.cvtColor(im,cv2.COLOR_BGR2GRAY)
print(bw_im.shape)
plt.imshow(bw_im, cmap='gray')

## Braille Image Processing

Agar hum text ko pehchaanne ke liye image classification lagana chahte hain, toh humein individual symbols ko crop karke unhe MNIST images jaisa banana hoga, jaisa humne pehle dekha hai. Yeh [Object Detection](../11-ObjectDetection/README.md) technique use karke kiya ja sakta hai, jise hum baad mein discuss karenge, lekin hum iske liye pure computer vision use karne ki bhi koshish kar sakte hain. Computer vision se characters ko alag karne ka acha description [is blog post](https://learnopencv.com/image-alignment-feature-based-using-opencv-c-python/) mein mil sakta hai - hum yahan sirf kuch computer vision techniques par focus karenge.

Sabse pehle, chaliye image ko thoda behtar banane ki koshish karte hain. Hum **thresholding** ka idea use kar sakte hain (jo [is OpenCV article](https://docs.opencv.org/4.x/d7/d4d/tutorial_py_thresholding.html) mein acchi tarah describe kiya gaya hai):

In [ ]:
im = cv2.blur(bw_im,(3,3))
im = cv2.adaptiveThreshold(im, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                           cv2.THRESH_BINARY_INV, 5, 4)
im = cv2.medianBlur(im, 3)
_,im = cv2.threshold(im, 0, 255, cv2.THRESH_OTSU)
im = cv2.GaussianBlur(im, (3,3), 0)
_,im = cv2.threshold(im, 0, 255, cv2.THRESH_OTSU)
plt.imshow(im)

Images ke saath kaam karne ke liye, humein individual dots ko "extract" karna hoga, yaani images ko individual dots ke coordinates ke set mein badalna hoga. Hum yeh **feature extraction** techniques use karke kar sakte hain, jaise SIFT, SURF ya [ORB](https://docs.opencv.org/4.x/d1/d89/tutorial_py_orb.html):

In [ ]:
orb = cv2.ORB_create(5000)
f,d = orb.detectAndCompute(im,None)
print(f"First 5 points: { [f[i].pt for i in range(5)]}")

In [ ]:
def plot_dots(dots):
    img = np.zeros((250,500))
    for x in dots:
        cv2.circle(img,(int(x[0]),int(x[1])),3,(255,0,0))
    plt.imshow(img)

pts = [x.pt for x in f]
plot_dots(pts)  

Poore text ko alag-alag letters mein todne ke liye, humein poore text ka bounding box jaanna hoga. Ise jaanne ke liye, hum minimum aur maximum coordinates calculate kar sakte hain:

In [ ]:
min_x, min_y, max_x, max_y = [int(f([z[i] for z in pts])) for f in (min,max) for i in (0,1)]
min_y+=13
plt.imshow(im[min_y:max_y,min_x:max_x])

Iske alawa, yeh text thoda ghuma hua ho sakta hai, aur ise perfectly align karne ke liye humein **perspective transformation** karna hoga. Hum points $(x_{min},y_{min}), (x_{min},y_{max}), (x_{max},y_{min}), (x_{max},y_{max})$ se define kiya gaya ek rectangle lenge aur ise proportional dimensions wali nayi image ke saath align karenge:

In [ ]:
off = 5
src_pts = np.array([(min_x-off,min_y-off),(min_x-off,max_y+off),
                    (max_x+off,min_y-off),(max_x+off,max_y+off)])
w = int(max_x-min_x+off*2)
h = int(max_y-min_y+off*2)
dst_pts = np.array([(0,0),(0,h),(w,0),(w,h)])
ho,m = cv2.findHomography(src_pts,dst_pts)
trim = cv2.warpPerspective(im,ho,(w,h))
plt.imshow(trim)

In [ ]:
char_h = 36
char_w = 24
def slice(img):
    dy,dx = img.shape
    y = 0
    while y+char_h<dy:
        x=0
        while x+char_w<dx:
            # Skip empty lines
            if np.max(img[y:y+char_h,x:x+char_w])>0:
                yield img[y:y+char_h,x:x+char_w]
            x+=char_w
        y+=char_h

sliced = list(slice(trim))
display_images(sliced)

## Frame Difference use karke Motion Detection

Video stream par motion detect karna ek bahut common task hai. Jaise, yeh humein tab alert paane deta hai jab surveillance camera par kuch hota hai. Agar hum samajhna chahte hain ki camera par kya ho raha hai, toh hum neural network use kar sakte hain - lekin neural network use karna tab zyada sasta padta hai jab humein pata ho ki kuch ho raha hai.

Motion detection ka main idea simple hai. Agar camera stable hai, toh camera se aane wale frames ek-doosre se kaafi similar hone chahiye. Kyunki frames ko arrays ke roop mein represent kiya jaata hai, toh do consecutive frames ke arrays ko subtract karne par humein pixel difference milega, jo stable frames ke liye kam hona chahiye aur jab image mein significant motion ho, toh yeh zyada ho jaayega.

Hum yeh seekhne se shuru karenge ki video ko kaise open karein aur ise frames ki series mein kaise badlein:

In [ ]:
vid = cv2.VideoCapture('data/motionvideo.mp4')

c = 0
frames = []
while vid.isOpened():
    ret, frame = vid.read()
    if not ret:
        break
    frames.append(frame)
    c+=1
vid.release()
print(f"Total frames: {c}")
display_images(frames[::150])

Kyunki motion detect karne ke liye color itna important nahi hai, hum saare frames ko grayscale mein convert kar denge. Phir, hum frames ka difference calculate karenge, aur unke norms ko plot karenge taaki activity ki matra visually dekhi ja sake:

In [ ]:
bwframes = [cv2.cvtColor(x,cv2.COLOR_BGR2GRAY) for x in frames]
diffs = [(p2-p1) for p1,p2 in zip(bwframes[:-1],bwframes[1:])]
diff_amps = np.array([np.linalg.norm(x) for x in diffs])
plt.plot(diff_amps)
display_images(diffs[::150],titles=diff_amps[::150])

Maan lijiye hum ek report banana chahte hain jo dikhaye ki camera ke saamne kya hua, har baar kuch hone par uchit image dikhakar. Yeh karne ke liye, hum shayad "event" ke start aur end frames ka pata lagana chahenge, aur beech ka frame display karna chahenge. Kuch noise hatane ke liye, hum upar diye gaye graph ko moving average function se smooth bhi karenge:

In [ ]:
def moving_average(x, w):
    return np.convolve(x, np.ones(w), 'valid') / w

threshold = 13000

plt.plot(moving_average(diff_amps,10))
plt.axhline(y=threshold, color='r', linestyle='-')

Ab hum `np.where` use karke un frames ka pata laga sakte hain jinme change ki matra threshold se zyada hai, aur 30 frames se lambi consecutive frames ki series nikaal sakte hain:

In [ ]:
active_frames = np.where(diff_amps>threshold)[0]

def subsequence(seq,min_length=30):
    ss = []
    for i,x in enumerate(seq[:-1]):
        ss.append(x)
        if x+1 != seq[i+1]:
            if len(ss)>min_length:
                return ss
            ss.clear()

sub = subsequence(active_frames)
print(sub)


In [ ]:
plt.imshow(frames[(sub[0]+sub[-1])//2])

In [ ]:
plt.imshow(cv2.cvtColor(frames[(sub[0]+sub[-1])//2],cv2.COLOR_BGR2RGB))

## Optical Flow use karke Motion nikaalna

Halaanki sirf do consecutive frames ko compare karne se humein change ki matra pata chalti hai, yeh humein yeh jaankari nahi deta ki actually kya aur kahan ho raha hai. Yeh jaankari paane ke liye, ek technique hai jise **[Optical Flow](https://docs.opencv.org/3.4/d4/dee/tutorial_optical_flow.html)** kaha jaata hai:

* **Dense Optical Flow** woh vector field banata hai jo dikhata hai ki har pixel kahan ja raha hai.
* **Sparse Optical Flow** image mein kuch specific features (jaise edges) leta hai aur frame se frame tak unki motion build karta hai.

Optical flow ke baare mein aur jaankari [is shaandaar tutorial](https://learnopencv.com/optical-flow-in-opencv/) mein padhiye.

Chaliye apne frames ke beech dense optical flow calculate karte hain:

In [ ]:
flows = [cv2.calcOpticalFlowFarneback(f1, f2, None, 0.5, 3, 15, 3, 5, 1.2, 0) 
         for f1,f2 in zip(bwframes[:-1],bwframes[1:])]
flows[0].shape

Jaisa ki aap dekh sakte hain, har frame ke liye flow frame ke dimensions aur 2 channels ke saath aata hai, jo optical flow vector ke x aur y components ke corresponding hain.

2D mein optical flow ko display karna thoda challenging ho sakta hai, lekin hum ek clever idea use kar sakte hain. Agar hum optical flow ko polar coordinates mein convert karein, toh humein har pixel ke liye do components milenge: *direction* aur *magnitude*. Hum magnitude ko pixel ki intensity se aur direction ko alag-alag colors se dikha sakte hain. Hum [HSV (Hue-Saturation-Value) color space](https://en.wikipedia.org/wiki/HSV_color_space) mein ek image banayenge, jahan hue direction se define hoga, value magnitude se, aur saturation 255 hoga.

In [ ]:
def flow_to_hsv(flow):
    hsvImg = np.zeros((flow.shape[0],flow.shape[1],3),dtype=np.uint8)
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsvImg[..., 0] = 0.5 * ang * 180 / np.pi
    hsvImg[..., 1] = 255
    hsvImg[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
    return cv2.cvtColor(hsvImg, cv2.COLOR_HSV2BGR)

start = sub[0]
stop = sub[-1]
print(start,stop)

frms = [flow_to_hsv(x) for x in flows[start:stop]]
display_images(frms[::25])

Toh, un frames mein green color ka matlab hai left ki taraf jaana, jabki blue color right ki taraf jaane ko dikhata hai.

Optical flow general direction mein motion ke baare mein conclusions nikaalne ke liye ek shaandaar tool ho sakta hai. Jaise, agar aap dekhte hain ki frame ke saare pixels kam ya zyada ek hi direction mein ja rahe hain - toh aap conclude kar sakte hain ki camera movement ho raha hai, aur ise balance karne ki koshish kar sakte hain.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->